# Local Notebook

Simple local workflow: load data, run inference before training, train a small LoRA adapter, compare the same rows after training.

All settings are in the next cell.

## 1. Config

In [1]:
from pathlib import Path

# Possible MODEL_NAME values:
# "HuggingFaceTB/SmolLM2-135M-Instruct"  # fastest local dry run
# "google/gemma-4-E2B-it"                # heavier local experiment
# "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "local_lora"

MAX_SEQ_LENGTH = 256
MAX_NEW_TOKENS = 32
# Use an integer for a small run, or None to use all rows not in validation.
TRAIN_ROWS = 1000
EVAL_ROWS = 32
PROBE_ROWS = 5
LOG_EVAL_POINTS = 8
SAVE_POINTS = 4

PROMPT_EXPERIMENT = "private_reasoning_boxed"
SYSTEM_PROMPTS = {
    "direct_raw_0_62": "Solve the puzzle and output only the final answer value. Do not explain. Do not write prefixes like 'answer:' or 'the answer is'.",
    "boxed_final": "Solve the puzzle carefully. Output only the final answer inside \\boxed{} with no trailing text.",
    "private_reasoning_boxed": "Solve the puzzle carefully. Reason internally if needed, but write only the final answer inside \\boxed{} with no trailing text.",
}
SYSTEM_PROMPT = SYSTEM_PROMPTS[PROMPT_EXPERIMENT]
# 'raw' reproduces the accepted baseline. 'boxed' is a separate metric-aligned experiment.
TRAIN_TARGET_FORMAT = "boxed"

LORA_R = 4
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
# For SmolLM/Gemma use ["q_proj", "v_proj"]. For Nemotron use ["in_proj", "out_proj"].
LORA_TARGET_MODULES = ["q_proj", "v_proj"]
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 32
EPOCHS = 1
LEARNING_RATE = 3e-4


## 2. Imports

In [2]:
import pathlib
import re
import time

import pandas as pd
import torch

# Windows TRL UTF-8 fix. Keep before importing TRL.
if not hasattr(pathlib.Path, "_nemotron_original_read_text"):
    pathlib.Path._nemotron_original_read_text = pathlib.Path.read_text

    def _read_text_utf8_by_default(self, encoding=None, errors=None):
        return pathlib.Path._nemotron_original_read_text(self, encoding=encoding or "utf-8", errors=errors)

    pathlib.Path.read_text = _read_text_utf8_by_default

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


C:\Users\mouak\PycharmProjects\kaggle_challenges\.venv\Lib\site-packages\torchao\quantization\quant_api.py:1745: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W0517 02:17:28.415000 28468 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## 3. Load data

In [3]:
train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

print("train:", train.shape)
print("test:", test.shape)
display(train.head())


train: (9500, 3)
test: (3, 2)


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


## 4. Prepare data

In [4]:
def infer_family(prompt: str) -> str:
    text = str(prompt).lower()
    if "bit manipulation" in text or "8-bit binary" in text:
        return "bit_manipulation"
    if "secret encryption" in text or "decrypt" in text:
        return "cipher"
    if "numeral system" in text:
        return "numeral"
    if "unit" in text and "convert" in text:
        return "unit_conversion"
    if "gravitational constant" in text or "falling distance" in text:
        return "gravity"
    if "transformation rules is applied to equations" in text or "determine the result for" in text:
        return "equation"
    return "unknown"


def normalize_answer(value) -> str:
    text = "" if value is None else str(value).strip()
    text = re.sub(r"^`+|`+$", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip(".").lower()


def extract_final_answer(text: str) -> str:
    """Extract a final answer using Kaggle's boxed-answer priority.

    The official metric first looks for answers in ``\\boxed{...}``. A
    literal ``}`` can be part of some answers, so this uses the last closing
    brace before the next boxed answer/end of text instead of stopping at the
    first ``}``.
    """
    text = "" if text is None else str(text)
    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    if boxed_starts:
        matches = []
        for idx, match in enumerate(boxed_starts):
            start = match.end()
            end = boxed_starts[idx + 1].start() if idx + 1 < len(boxed_starts) else len(text)
            segment = text[start:end]
            last_brace = segment.rfind("}")
            matches.append(segment[:last_brace] if last_brace != -1 else segment)
        non_empty = [match.strip() for match in matches if match.strip()]
        return normalize_answer(non_empty[-1] if non_empty else matches[-1])

    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
        r"final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return normalize_answer(matches[-1])

    number_matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if number_matches:
        return normalize_answer(number_matches[-1])

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        line = re.sub(r"^(final answer|answer|assistant)\s*[:\-]\s*", "", line, flags=re.I).strip()
        if line:
            return normalize_answer(line.strip("`").strip())
    return normalize_answer(text)


def format_training_answer(answer: str) -> str:
    """Return the assistant target text for SFT examples.

    Use raw to reproduce the accepted 0.62 Colab baseline, where the
    model is trained on the short answer exactly as it appears in train.csv.
    Use boxed only for an explicit experiment aligned to Kaggle's metric
    prompt/extractor.
    """
    answer_text = "" if answer is None else str(answer).strip()
    if TRAIN_TARGET_FORMAT == "raw":
        return answer_text
    if TRAIN_TARGET_FORMAT == "boxed":
        return f"\\boxed{{{normalize_answer(answer_text)}}}"
    raise ValueError(f"Unsupported TRAIN_TARGET_FORMAT: {TRAIN_TARGET_FORMAT!r}")


def build_sft_text(prompt: str, answer: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n{format_training_answer(answer)}"


def build_prompt(prompt: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n"


train = train.copy()
train["family"] = train["prompt"].map(infer_family)

sft = train[["id", "family", "prompt", "answer"]].copy()
sft["text"] = [build_sft_text(p, a) for p, a in zip(sft["prompt"], sft["answer"])]

if EVAL_ROWS and EVAL_ROWS > 0:
    valid = sft.sample(n=min(EVAL_ROWS, len(sft) - 1), random_state=42)
    train_sft = sft.drop(index=valid.index)
else:
    valid = sft.iloc[0:0].copy()
    train_sft = sft.copy()
if TRAIN_ROWS is not None:
    train_sft = train_sft.head(TRAIN_ROWS)
probe_questions = sft.groupby("family", group_keys=False).head(1).head(PROBE_ROWS).reset_index(drop=True)

print(train["family"].value_counts())
print("train rows:", len(train_sft), "valid rows:", len(valid))
display(probe_questions[["id", "family", "answer"]])


family
bit_manipulation    1602
gravity             1597
unit_conversion     1594
cipher              1576
numeral             1576
equation            1555
Name: count, dtype: int64
train rows: 1000 valid rows: 32


,id,family,answer
0,00066667,bit_manipulation,10010111
1,00189f6a,cipher,cat imagines book
2,001b24c4,numeral,XXXVIII
3,00208201,unit_conversion,16.65
4,0040ff76,gravity,154.62


## 5. Load model

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("model:", MODEL_NAME)
print("device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE, trust_remote_code=True).to(DEVICE)
model.eval()


model: HuggingFaceTB/SmolLM2-135M-Instruct
device: cpu


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (r

## 6. Module summary

In [6]:
def dtype_name(dtype):
    return str(dtype).replace("torch.", "") if dtype is not None else "-"


module_rows = []
for name, module in model.named_modules():
    class_name = module.__class__.__name__
    if "Linear" in class_name or "Conv" in class_name:
        short_name = name.split(".")[-1]
        weight = getattr(module, "weight", None)
        compute_dtype = getattr(module, "compute_dtype", None)
        param_dtypes = sorted({dtype_name(p.dtype) for p in module.parameters(recurse=False)})
        buffer_dtypes = sorted({dtype_name(b.dtype) for b in module.buffers(recurse=False)})
        module_rows.append({
            "module": short_name,
            "class": class_name,
            "weight_dtype": dtype_name(getattr(weight, "dtype", None)),
            "compute_dtype": dtype_name(compute_dtype),
            "param_dtypes": ", ".join(param_dtypes) or "-",
            "buffer_dtypes": ", ".join(buffer_dtypes) or "-",
        })

module_summary = (
    pd.DataFrame(module_rows)
    .groupby(["module", "class", "weight_dtype", "compute_dtype", "param_dtypes", "buffer_dtypes"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(module_summary)
print("LoRA targets:", LORA_TARGET_MODULES)


,module,class,weight_dtype,compute_dtype,param_dtypes,buffer_dtypes,count
0,down_proj,Linear,float32,-,float32,-,30
1,gate_proj,Linear,float32,-,float32,-,30
2,k_proj,Linear,float32,-,float32,-,30
4,o_proj,Linear,float32,-,float32,-,30
6,up_proj,Linear,float32,-,float32,-,30
5,q_proj,Linear,float32,-,float32,-,30
7,v_proj,Linear,float32,-,float32,-,30
3,lm_head,Linear,float32,-,float32,-,1


LoRA targets: ['q_proj', 'v_proj']


## 7. Inference before training

In [7]:
def run_inference(current_model, rows, name):
    current_model.eval()
    records = []
    for row in rows.itertuples(index=False):
        inputs = tokenizer(build_prompt(row.prompt), return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(DEVICE)
        start = time.time()
        with torch.no_grad():
            output_ids = current_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        seconds = time.time() - start
        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        pred = extract_final_answer(raw)
        records.append({
            "id": row.id,
            "family": row.family,
            "gold": row.answer,
            name: pred,
            f"{name}_match": normalize_answer(row.answer) == normalize_answer(pred),
            f"{name}_seconds": seconds,
            f"{name}_raw": raw,
        })
    return pd.DataFrame(records)

before_results = run_inference(model, probe_questions, "before")
pd.set_option("display.max_colwidth", 240)
display(before_results[["id", "family", "gold", "before", "before_match", "before_seconds", "before_raw"]])


,id,family,gold,before,before_match,before_seconds,before_raw
0,00066667,bit_manipulation,10010111,00111111,False,1.519831,"11111\n\nOutput:\n00111111\n\nUser:\nIn Alice's Wonderland, a secret bit"
1,00189f6a,cipher,cat imagines book,"first, let's break",False,1.310527,"The puzzle is to decrypt the text ""trb wzrswvog hffk"" using the encryption rules.\n\nFirst, let's break"
2,001b24c4,numeral,XXXVIII,38,False,0.912688,38 = 11 + 94\n\nThe correct answer is 38.
3,00208201,unit_conversion,16.65,35.85,False,1.336298,10.08 m becomes 6.69\n17.83 m becomes 11.83\n35.85
4,0040ff76,gravity,154.62,0.5,False,1.368336,"To solve this problem, we need to find the falling distance for t = 4.41s given d = 0.5*g*t"


## 8. Train LoRA

In [ ]:
import math

train_ds = Dataset.from_pandas(train_sft[["text"]], preserve_index=False)
valid_ds = Dataset.from_pandas(valid[["text"]], preserve_index=False) if len(valid) else None

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

steps_per_epoch = max(1, math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM_STEPS)))
total_train_steps = max(1, math.ceil(steps_per_epoch * float(EPOCHS)))
logging_steps = max(1, total_train_steps // LOG_EVAL_POINTS)
eval_steps = logging_steps
save_steps = max(logging_steps, total_train_steps // SAVE_POINTS)
eval_strategy = "steps" if valid_ds is not None else "no"
print(
    "train steps:", total_train_steps,
    "logging_steps:", logging_steps,
    "eval_steps:", eval_steps if eval_strategy != "no" else None,
    "save_steps:", save_steps,
)

args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=logging_steps,
    logging_first_step=True,
    save_steps=save_steps,
    eval_strategy=eval_strategy,
    eval_steps=eval_steps if eval_strategy != "no" else None,
    report_to="tensorboard",
    fp16=(DEVICE == "cuda"),
    bf16=False,
    use_cpu=(DEVICE == "cpu"),
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)
trainer.train()


train steps: 32 logging_steps: 4 eval_steps: 4 save_steps: 8


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
4,3.029717,2.481994
8,2.986412,2.391275


KeyboardInterrupt: 

## 9. Compare after training

In [ ]:
after_results = run_inference(trainer.model, probe_questions, "after")
comparison = before_results.merge(after_results, on=["id", "family", "gold"])
display(comparison[["id", "family", "gold", "before", "before_match", "before_raw", "after", "after_match", "after_raw"]])


## 10. Save LoRA adapter

This saves the trained adapter. For SmolLM this is only a packaging dry run; Kaggle scoring requires a Nemotron-compatible adapter.

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / "adapter"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(ADAPTER_DIR))

print("saved adapter to:", ADAPTER_DIR)
display(pd.DataFrame({"file": sorted(p.name for p in ADAPTER_DIR.iterdir() if p.is_file())}))


## 11. Build `submission.zip`

Kaggle expects a zip containing the required LoRA adapter files at the zip root: `adapter_config.json` and `adapter_model.safetensors`.

In [ ]:
import zipfile

SUBMISSION_ZIP = OUTPUT_DIR / "submission.zip"
required_files = [ADAPTER_DIR / "adapter_config.json", ADAPTER_DIR / "adapter_model.safetensors"]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing adapter files: " + ", ".join(missing_files))

if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in required_files:
        zf.write(path, path.name)

with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    zip_files = zf.namelist()

print("created:", SUBMISSION_ZIP)
print("size MB:", round(SUBMISSION_ZIP.stat().st_size / 1024 / 1024, 2))
display(pd.DataFrame({"zip_file": zip_files}))


## 12. Optional Kaggle submit

Keep this disabled until you really want to spend a daily submission. A SmolLM adapter is not compatible with the Nemotron metric, so the guard blocks it by default.

In [ ]:
SUBMIT_TO_KAGGLE = False
ALLOW_NON_NEMOTRON_SUBMISSION = False
KAGGLE_COMPETITION = "nvidia-nemotron-model-reasoning-challenge"
KAGGLE_MESSAGE = f"adapter test from {MODEL_NAME}"

is_nemotron_adapter = MODEL_NAME == "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

if SUBMIT_TO_KAGGLE:
    if not is_nemotron_adapter and not ALLOW_NON_NEMOTRON_SUBMISSION:
        raise RuntimeError(
            "This adapter was trained from MODEL_NAME=" + repr(MODEL_NAME) + " and will not match Kaggle's Nemotron base model. "
            "Set ALLOW_NON_NEMOTRON_SUBMISSION=True only if you intentionally want to test a failing upload."
        )
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ImportError as exc:
        raise ImportError("Install Kaggle API first: !pip install kaggle") from exc

    api = KaggleApi()
    api.authenticate()
    api.competition_submit(str(SUBMISSION_ZIP), KAGGLE_MESSAGE, KAGGLE_COMPETITION)
    print("submitted", SUBMISSION_ZIP, "to", KAGGLE_COMPETITION)
else:
    print("dry run only; set SUBMIT_TO_KAGGLE=True to submit")
    print("zip ready:", SUBMISSION_ZIP)
